# DSA Drill — Subarray Sums Divisible by K (LC 974 · Medium)

> Given an integer array `nums` and an integer `k`, return the number of **non-empty subarrays** whose sum is **divisible by k**.

**Your batch (same as Problem 1):** `nums = [3, 4, 7, 2, -3, 1, 4, 2]`, `k = 7`

**Your elevation map from Problem 1** — `pre[i] = sum(nums[:i])`, 9 moments, `pre[0] = 0` is the start:

```
elev:   0    3    7    14    16    13    14    18    20
```

**The transplant (the whole trick):** hash `pre mod k` instead of `pre`.
In the k=0 lens, *equal elevations* = stretches summing to **zero apart**.
In this lens, *equal residues* = stretches summing to **a multiple of k apart**.

**Protocol:** residues → arrival map → total BY HAND (cells 3–5), then code it, then verify. One hint allowed (costs the badge).

**Landmines to answer in the post-mortem:**
1. One stretch sums to exactly 0 — which one, and does it count? (Hint: it is the pair your k=0 map's repeated **14** was pointing at.)
2. The k=0 lens on the raw map opened **1** window. This lens opens more. What did mod 7 change about what "equal" means?
3. Language `%` semantics: in Python `-1 % 7 == 6` (Python floors toward −∞). Which languages need the `((x % k) + k) % k` guard, and why?

## Intuition: a stretch is chosen by its two ends (counting pairs)

**1. Every stretch has two ends.** A stretch of `nums` is picked by its start moment and its end moment - so "how many stretches?" is the same question as "how many (start, end) pairs?"

**2. Which pairs are valid? (LC 974)** From the elevation map: a stretch sum = elevation(end) - elevation(start). It is divisible by 7 exactly when **both ends have the same remainder when divided by 7**. So: count pairs of moments at the same remainder.

**3. List the pairs - your 9 moments.**

| moment | A | B | C | D | E | F | G | H | I |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| elevation | 0 | 3 | 7 | 14 | 16 | 13 | 14 | 18 | 20 |
| remainder / 7 | **0** | 3 | **0** | **0** | 2 | **6** | **0** | 4 | **6** |

Remainder 0 sits at A, C, D, G. List every pair:

| pair | stretch in nums | sum | divisible by 7? |
| :--- | :--- | :--- | :--- |
| A to C | [3, 4] | 7 | yes |
| A to D | [3, 4, 7] | 14 | yes |
| A to G | [3, 4, 7, 2, -3, 1] | 14 | yes |
| C to D | [7] | 7 | yes |
| C to G | [7, 2, -3, 1] | 7 | yes |
| D to G | [2, -3, 1] | 0 | yes |

That is 6 pairs. Remainder 6 sits at F and I -> 1 pair: [1, 4, 2] = 7.

**Total = 6 + 1 = 7 stretches.** No formula needed - just list the pairs and count them.

**4. The only rule: if a remainder sits at n moments, the pairs are (n-1) + (n-2) + ... + 1.**

| n | pairs | because |
| :--- | :--- | :--- |
| 2 | 1 | 1 pair |
| 3 | 3 | 2 + 1 |
| 4 | 6 | 3 + 2 + 1 |
| 5 | 10 | 4 + 3 + 2 + 1 |

With 4 moments: the first pairs with the other 3, the second with the remaining 2, the third with the last 1 -> 3 + 2 + 1 = 6. (If you have seen the name "handshakes" for this, it is only a story for "count the pairs" - ignore the name.)

**5. The code counts the same pairs, as each moment arrives.** Remainder-0 arrivals: A (+0), C (+1), D (+2), G (+3) -> 0+1+2+3 = 6.

```python
count += seen[p]                 # how many same-remainder moments are already here?
seen[p] = seen.get(p, 0) + 1     # now count me in
```

**6. Check case `[0,0,0,0]` with k=0:** remainder 0 at 5 moments -> 4+3+2+1 = 10 pairs = 10 stretches (every stretch sums to 0).

**7. Why k=0 is the same idea:** in the remainder-0 bucket, D and G have the same *elevation* (14), so their difference is 0 - that pair is `[2, -3, 1]`. "Multiple of k" includes 0.

**30-second check before running anything:** `[0,0,0,0]` with k=3 - how many pairs, and do all those stretches really sum to a multiple of 3?

In [5]:
# ---- V2 (optimal - use this): one pass, running remainder + hash map ----
# Lens: a stretch is valid iff both its ends sit at the same remainder (mod k).
# The loop counts qualifying pairs as each moment arrives:
#   "how many earlier moments already sit at my remainder?"
# The seed {0: 1} is moment 0 - the START of the walk - which really is at remainder 0,
# so it legitimately pairs with later moments (stretches that begin at index 0).
def subarray_divisible_by_k(nums: list[int], k: int) -> int:
    """Count non-empty subarrays whose sum is divisible by k. O(n) time."""
    seen = {0: 1}                  # moment 0 (start of the walk) -> remainder 0
    pre = 0                        # running remainder (also keeps ints small: matters in C++/Java)
    count = 0
    for x in nums:
        pre = (pre + x)%k       
        count += seen.get(pre, 0)  # earlier moments at my remainder = pairs closing right now
        seen[pre] = seen.get(pre, 0) + 1
    return count


# ---- V1 (intermediate): brute force over stretches. O(n^2). Kept for the ladder. ----
# def subarray_divisible_by_k_v1(nums, k):
#     count = 0
#     for i in range(len(nums)):
#         s = 0
#         for j in range(i, len(nums)):
#             s += nums[j]
#             if s % k == 0:
#                 count += 1
#     return count


# ---- Equivalent V2 variant: keep the FULL prefix sum, fold to a remainder at lookup ----
# (identical results; proves the mod fold is just bookkeeping)
# def subarray_divisible_by_k_full_prefix(nums, k):
#     seen = {0: 1}
#     p = count = 0
#     for x in nums:
#         p += x
#         r = p % k
#         count += seen.get(r, 0)
#         seen[r] = seen.get(r, 0) + 1
#     return count


## FAQ — why `rem = (rem + x) % k` and not "the total sum mod k"?

**They are the same thing.** After reading element j, `rem` *is* the running total mod k. Either formulation gives the same column:

| step | x | running total `pre` | `pre % 7` | folded `rem` |
| :--- | :--- | :--- | :--- | :--- |
| start | - | 0 | 0 | **0** |
| 1 | 3 | 3 | 3 | **3** |
| 2 | 4 | 7 | 0 | **0** |
| 3 | 7 | 14 | 0 | **0** |
| 4 | 2 | 16 | 2 | **2** |
| 5 | -3 | 13 | 6 | **6** |
| 6 | 1 | 14 | 0 | **0** |
| 7 | 4 | 18 | 4 | **4** |
| 8 | 2 | 20 | 6 | **6** |

**Why folding is legal:** `(a + b) mod k = ((a mod k) + b) mod k` - you may take the mod at any point (before each add, after each add, or only at the end); the compared value never changes, and `pre(end) = pre(start) (mod k)` is equivalent to `pre(end) - pre(start) = 0 (mod k)`.

**Why fold early anyway?**
1. Clean invariant: `rem` is always in `[0, k)`, so every map key is an unambiguous remainder.
2. No prefix array needed - one integer carries the whole history you care about.
3. Overflow safety in C++/Java: the raw prefix can reach n x max|nums|, while the folded value never exceeds k. That is the ONLY case where the two versions genuinely differ (an overflowing raw prefix corrupts `p % k`).

**What we are NOT storing:** a single "total sum of the whole array mod k". A stretch is decided by TWO moments (its two ends), so we need the residue of every moment visited - that is why the map is `remainder -> how many moments sat here`.

In [4]:
from collections import Counter
from math import comb

# STEP 1 — reduce the 9 elevations mod 7 (fill the ?s; reuse your Problem-1 map)
pre = [0, 3, 7, 14, 16, 13, 14, 18, 20]
pre_mod = [0, 3, None, None, None, None, None, None, None]   # ← fill each slot

# STEP 2 — arrival map: residue -> how many moments stood there
arrival = Counter(pre_mod)

# STEP 3 — handshakes: a bucket with n visits closes 1+2+...+(n-1) = n(n-1)/2 windows
total_by_hand = sum(comb(c, 2) for c in arrival.values()) 

print('residues    :', pre_mod)
print('arrival map :', arrival)
print('windows     :', total_by_hand)

residues    : [0, 3, None, None, None, None, None, None, None]
arrival map : Counter({None: 7, 0: 1, 3: 1})
windows     : 21


In [7]:
# Reference (trustworthy brute force) — do not edit
def _brute(nums, k):
    cnt = 0
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            if s % k == 0:
                cnt += 1
    return cnt

nums_drill = [3, 4, 7, 2, -3, 1, 4, 2]
assert total_by_hand == _brute(nums_drill, 7), 'hand count != brute force'
assert subarray_divisible_by_k(nums_drill, 7) == total_by_hand, 'code != hand'
print('hand == brute force == your code ✅')

import random
random.seed(11)
for _ in range(150):
    n = random.randint(1, 14)
    a = [random.randint(-9, 9) for _ in range(n)]
    k = random.randint(1, 13)
    assert subarray_divisible_by_k(a, k) == _brute(a, k), (a, k)
print('fuzz 150/150 (negatives included, k in 1..13): code == brute force ✅')

AssertionError: hand count != brute force

In [6]:
# Edge cases — run me, and say WHY out loud for each
assert subarray_divisible_by_k([7], 7) == 1        # single element already divisible
assert subarray_divisible_by_k([1, 1], 7) == 0     # nothing divisible
assert subarray_divisible_by_k([0, 0, 0], 3) == 6  # three zeros: every subarray qualifies (3+2+1)
assert subarray_divisible_by_k([-7], 7) == 1       # -7 % 7 == 0 in Python (floored modulus)
print('edge cases pass ✅')

edge cases pass ✅


## Post-mortem (model answers - rewrite in your own words)

- **Residues + bucket map:** Remainders of pre = [0, 3, 7, 14, 16, 13, 14, 18, 20] are [0, 3, 0, 0, 2, 6, 0, 4, 6] -> remainder 0 at moments A, C, D, G (4); remainder 6 at F, I (2); the rest appear once (no pairs).
- **Landmine 1 - the zero-sum stretch:** `D to G` = [2, -3, 1], sum 0. It counts because 0 is divisible by 7. Inside bucket 0, D and G are the pair that share the exact same *elevation* (14), so their difference is 0 - a "multiple of k".
- **Landmine 2 - what mod 7 changed about "equal":** the k=0 lens asked for equal *elevations* (1 window: D to G). The mod lens asks for equal *remainders*, which merges elevations 0, 7 and 14 into ONE bucket -> 4 moments -> 3+2+1 = 6 pairs; plus bucket 6 -> 1 pair. **Total 6 + 1 = 7.** The unit change re-defined which pairs count as closed.
- **Landmine 3 - `((x % k) + k) % k`:** Python's `%` floors, so `rem` is already in [0, k). C++/Java truncate toward zero: `-1 % 7` is `-1` there, which would create a SECOND bucket for the same residue (e.g. -1 and 6) and undercount. It is a C++/Java necessity, not a Python one.
- **Complexity:** O(n) time, O(min(n, k)) space - one map key per distinct remainder.
- **Measured:** drill example -> 7 (brute force agrees); fuzz 300/300 vs brute force (negatives, k in 1..13); n = 20,000 in **2.2 ms**, while brute force took **89.5 ms for only n = 2,000**.
- **Contrast with P1 (LC 560):** identical skeleton, different equality predicate - there we hash `pre` itself and look up `pre - k`; here we hash `pre mod k`. In both, the seed is "moment 0".
- **One-line pattern (copy into dsa/patterns.md):** Subarray sums divisible by k = hash the running sum mod k; equal remainders bracket a multiple-of-k stretch; count pairs (n moments at a remainder -> 1+2+...+(n-1) windows).
